# 🌿 Vegetation Stress Early Warning System
### AI AgriVision · Protected Areas Module · Egypt

> A dedicated early-warning module for detecting vegetation stress across Egypt's 31 protected areas, built on real satellite and reanalysis data — designed as the foundation for future integration into the AI AgriVision platform.

---

## System Architecture

| Stage | Purpose | Method |
|-------|---------|--------|
| **1. Data Loading** | Ingest real ERA5 + Landsat observations | 31 CSV files from Google Drive |
| **2. Label Generation** | Derive stress labels from observed signal | NDVI anomaly detection — not rule-generated |
| **3. Feature Engineering** | Build ~55 predictive features | ERA5 climate + NDVI indices + soil moisture |
| **3b. Area Baselines** | Per-area climate statistics | Precipitation median/std · Soil moisture P20 |
| **4. EWS v1 Rules** | Interpretable safety layer | Recalibrated anomaly-based rule engine |
| **5. ML Ensemble** | Probabilistic stress prediction | XGBoost + Random Forest + Logistic Regression |
| **5b. VIF Check** | Collinearity control for LR | Variance Inflation Factor analysis |
| **6. Fusion** | Combine rule-based + ML outputs | 3-branch weighted fusion with dual thresholds |

---

## Dataset

| Property | Value |
|----------|-------|
| **Source** | Google Earth Engine (Landsat 4/5/7/8) + ERA5 Reanalysis |
| **Variables** | NDVI, SAVI, EVI, temperature, precipitation, soil moisture, radiation, wind |
| **Period** | 1983 – 2025 |
| **Coverage** | 31 Egyptian protected areas |
| **Format** | One CSV per protected area (subject 0 – 30) |

---

> **Execution order:** Run all cells sequentially from top to bottom.

---
## Section 1 — Setup: Installations, Imports & Logging

In [1]:
# ── Installations (run once) ───────────────────────────────────────────────────
# !pip install xgboost scikit-learn statsmodels shap pandas numpy requests joblib

In [2]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
# ── Standard library ──────────────────────────────────────────────────────────
import os, json, logging, warnings, datetime as dt, glob
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    confusion_matrix, classification_report,
    roc_auc_score, balanced_accuracy_score
)
from statsmodels.stats.outliers_influence import variance_inflation_factor
import xgboost as xgb
import shap, joblib

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('ews_protected_areas')
logger.info('EWS Protected Areas v2.2 notebook initialised')

---
## Section 2 — Constants & Configuration

All system-wide parameters are defined here — vegetation stress thresholds, fusion decision boundaries, model hyperparameters, and per-area baseline dictionaries that are populated after data loading.

In [4]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DRIVE_DATA_PATH = '/content/drive/MyDrive/data_EWS/data/'

# ── Artifacts are saved INSIDE Google Drive (not the temporary Colab session) ──
# This makes the trained models persistent: they survive runtime resets/
# disconnects and can be reloaded later without retraining (see Section 13).
ARTIFACTS_DIR = Path('/content/drive/MyDrive/data_EWS/artifacts_protected')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

PROTECTED_AREAS = {
    0 : 'محمية اشتوم الجميل',
    1 : 'محمية البرلس',
    2 : 'محمية الجزر الشمالية للبحر الأحمر',
    3 : 'محمية الجلف الكبير',
    4 : 'محمية الدبابية',
    5 : 'محمية الزرانيق',
    6 : 'محمية الصحراء البيضاء',
    7 : 'محمية العميد',
    8 : 'محمية الغابة المتحجرة',
    9 : 'محمية الواحات البحرية – الجزء الشرقي',
    10: 'محمية الواحات البحرية – الجزء الغربي',
    11: 'محمية الواحات البحرية – الجزء الوسطي',
    12: 'محمية رأس محمد',
    13: 'محمية سالوجا وغزال',
    14: 'محمية سانت كاثرين',
    15: 'محمية سيوة – القطاع الأوسط الجنوبي',
    16: 'محمية سيوة – القطاع الغربي',
    17: 'محمية سيوة – القطاع الشرقي',
    18: 'محمية طابا',
    19: 'محمية علبة',
    20: 'محمية قارون',
    21: 'محمية قبة الحسنة',
    22: 'محمية كهف سنور',
    23: 'محمية نبق',
    24: 'محمية نيزك جبل كامل',
    25: 'محمية وادي الأسيوطي',
    26: 'محمية وادي الجمال',
    27: 'محمية وادي الريان',
    28: 'محمية وادي العلاقي',
    29: 'محمية وادي دجلة',
    30: 'محمية أبو جالوم',
}
AREA_IDX = {v: k for k, v in PROTECTED_AREAS.items()}

# ── Per-area climate baselines (populated in Section 3b after data load) ─────
# Used by v2.3 anomaly-based flags to avoid penalising chronic desert conditions
AREA_PRECIP_BASELINES   : Dict[int, float] = {}   # median precipitation per area
AREA_PRECIP_STDS        : Dict[int, float] = {}   # std precipitation per area
AREA_SOILMOIST_P20      : Dict[int, float] = {}   # 20th-percentile soil moisture per area

# ── NDVI Stress Detection Parameters ─────────────────────────────────────────
NDVI_STRESS = dict(
    rolling_window   = 30,
    stress_threshold = 1.5,
    min_ndvi_valid   = -0.1,
    max_ndvi_valid   =  0.9,
    noise_rate       = 0.03,
)

# ── Vegetation Stress Thresholds (values in CELSIUS after conversion) ─────────
# Reference: UNEP (2006) Global Deserts Outlook; FAO (2010) Global Forest Resources
VEG_STRESS = dict(
    temp_opt_low    = 15.0,    # °C
    temp_opt_high   = 35.0,    # °C
    temp_critical   = 42.0,    # °C — heat stress for desert vegetation
    soil_moist_low  = 0.05,    # m³/m³
    precip_dry      = 1.0,     # mm/day
    radiation_high  = 25.0,    # MJ/m²
    evap_stress     = 5.0,     # mm/day
    ndvi_healthy    = 0.3,
    ndvi_stressed   = 0.15,
)

# ── Fusion thresholds ─────────────────────────────────────────────────────────
# Fusion branch routing thresholds — calibrated for 7.9% positive rate.
# Calibrated model probs for true negatives cluster in [0.03–0.17];
# thresholds are set to keep all three branches active.
THRESH_HIGH_CONF  = 0.55    # high-confidence ML branch activation threshold
THRESH_LOW_CONF   = 0.10    # below this → v1 fallback (avoids ML silence on imbalanced data)
ENSEMBLE_DISAGREE = 0.10

# Production threshold (best-F2, recomputed after training in Section 8)
DECISION_THRESHOLD = 0.571   # placeholder — overwritten after training
SAFETY_THRESHOLD   = 0.018   # recall≥0.85 — safety audit mode only

# ── Weighted Ensemble Aggregation ────────────────────────────────────────────
# LR produces systematically higher calibrated probabilities than the tree models
# on this imbalanced dataset (7.9% positive rate). Down-weighting LR prevents it
# from dominating the ensemble mean and pushing predictions above fusion thresholds.
ENSEMBLE_WEIGHTS = {
    'xgb': 0.5,   # highest PR-AUC — primary signal
    'rf' : 0.3,   # strong diversity from XGB
    'lr' : 0.2,   # inflated probs on imbalanced data — reduced influence
}

# ── Model hyper-parameters ────────────────────────────────────────────────────
XGB_PARAMS = dict(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=1,   # recomputed per-split in train_ensemble() — not mutated globally
    use_label_encoder=False, eval_metric='logloss',
    random_state=RANDOM_SEED, n_jobs=-1
)
RF_PARAMS = dict(
    n_estimators=300, max_depth=12, min_samples_leaf=5,
    class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1
)
LR_PARAMS = dict(C=0.5, max_iter=1000, random_state=RANDOM_SEED, n_jobs=-1)

logger.info('Config loaded — protected_areas=%d', len(PROTECTED_AREAS))

---
## Section 3 — Data Loading

Loads all 31 protected-area CSV files from Google Drive and concatenates them into a single DataFrame.

| Data Source | Variables | Notes |
|-------------|-----------|-------|
| **Landsat 4/5/7/8** via Google Earth Engine | NDVI, SAVI, EVI, vegetation cover | Seasonal composites |
| **ERA5 Reanalysis** | Temperature (Kelvin), precipitation, evaporation, wind, radiation | Converted to Celsius in Section 5 |
| **ERA5 Soil** | Volumetric soil water content — 4 depth layers | Used for water stress detection |

> Each CSV file represents one protected area (subjects 0 – 30).

In [5]:
def load_protected_areas_data(data_path: str = DRIVE_DATA_PATH) -> pd.DataFrame:
    """Load all 31 CSV files from Google Drive and concatenate into one DataFrame."""
    csv_files = sorted(glob.glob(os.path.join(data_path, '*.csv')))
    if not csv_files:
        raise FileNotFoundError(
            f'No CSV files found at {data_path}\n'
            f'تأكد إن Google Drive مماونت صح والباث صح:\n'
            f'  /content/drive/MyDrive/data_EWS/data/'
        )
    logger.info('Found %d CSV files', len(csv_files))
    dfs = []
    for fpath in csv_files:
        fname = os.path.basename(fpath)
        try:
            area_id = int(fname.replace('.csv', '').replace('subject_', '').strip())
        except ValueError:
            logger.warning('Could not parse area_id from: %s — skipping', fname)
            continue
        df = pd.read_csv(fpath)
        df['area_id']   = area_id
        df['area_name'] = PROTECTED_AREAS.get(area_id, f'Area_{area_id}')
        dfs.append(df)
        logger.info('  Loaded: %s → %d rows | area: %s',
                    fname, len(df), PROTECTED_AREAS.get(area_id, '?'))
    combined = pd.concat(dfs, ignore_index=True)
    logger.info('Total records: %d | areas: %d', len(combined), combined['area_id'].nunique())
    return combined


RAW_DF = load_protected_areas_data()

print(f'Shape           : {RAW_DF.shape}')
print(f'Columns         : {list(RAW_DF.columns)}')
print(f'Areas loaded    : {RAW_DF["area_id"].nunique()}')
print(f'Date range      : {RAW_DF["day_count"].min()} → {RAW_DF["day_count"].max()}')
RAW_DF.head(3)

Shape           : (5316, 31)
Columns         : ['pa_name', 'season', 'year', 'ndvi_mean', 'evi', 'savi', 'vegetation_percent', 'sensor', 'day_count', 'temperature_2m', 'total_precipitation_sum', 'u_component_of_wind_10m', 'v_component_of_wind_10m', 'surface_pressure', 'dewpoint_temperature_2m', 'runoff_sum', 'skin_temperature', 'soil_temperature_level_1', 'soil_temperature_level_2', 'soil_temperature_level_3', 'soil_temperature_level_4', 'sub_surface_runoff_sum', 'surface_runoff_sum', 'surface_solar_radiation_downwards_sum', 'total_evaporation_sum', 'volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_2', 'volumetric_soil_water_layer_3', 'volumetric_soil_water_layer_4', 'area_id', 'area_name']
Areas loaded    : 31
Date range      : 76 → 92


,pa_name,season,year,ndvi_mean,evi,savi,vegetation_percent,sensor,day_count,temperature_2m,...,sub_surface_runoff_sum,surface_runoff_sum,surface_solar_radiation_downwards_sum,total_evaporation_sum,volumetric_soil_water_layer_1,volumetric_soil_water_layer_2,volumetric_soil_water_layer_3,volumetric_soil_water_layer_4,area_id,area_name
0,محمية اشتوم الجميل,0,1984,0.002778,-0.003418,0.004167,0.050435,L5,91,287.998343,...,0.0,2.562836e-06,1.288838e+07,-0.000337,0.027511,-2.839414e-22,0.067505,0.080719,0,محمية اشتوم الجميل
1,محمية اشتوم الجميل,1,1984,0.017579,-0.024366,0.026368,0.167890,L5,92,291.967904,...,0.0,1.354548e-06,5.256857e+04,-0.000194,0.010445,-6.860835e-22,0.067367,0.080719,0,محمية اشتوم الجميل
2,محمية اشتوم الجميل,2,1984,0.027276,-0.035382,0.040914,0.264349,L5,92,298.345279,...,0.0,1.724971e-07,2.695310e+07,-0.000063,0.001116,-9.199955e-22,0.063710,0.080719,0,محمية اشتوم الجميل


In [6]:
# ── Data quality check ────────────────────────────────────────────────────────
print('=== Data Quality Report ===')
print(f'Total rows     : {len(RAW_DF):,}')
nulls = RAW_DF.isnull().sum()
print('Null values    :', 'None ✅' if not nulls.any() else '')
if nulls.any(): print(nulls[nulls > 0].to_string())
print()
print('NDVI stats:')
print(RAW_DF[['ndvi_mean', 'savi', 'evi']].describe().round(4))
print()
print('Climate stats (temperature in Kelvin from ERA5):')
print(RAW_DF[['temperature_2m', 'total_precipitation_sum',
             'volumetric_soil_water_layer_1']].describe().round(4))
print()
# Confirm temperature is in Kelvin
t_mean = RAW_DF['temperature_2m'].mean()
print(f'Mean temperature_2m = {t_mean:.1f} K = {t_mean - 273.15:.1f} °C  ← ERA5 Kelvin confirmed')

=== Data Quality Report ===
Total rows     : 5,316
Null values    : None ✅

NDVI stats:
       ndvi_mean       savi        evi
count  5316.0000  5316.0000  5316.0000
mean      0.0362     0.0543    -0.0404
std       0.0278     0.0417     0.0307
min      -0.1748    -0.2621    -0.3142
25%       0.0191     0.0286    -0.0583
50%       0.0407     0.0611    -0.0427
75%       0.0520     0.0779    -0.0201
max       0.1934     0.2901     0.2855

Climate stats (temperature in Kelvin from ERA5):
       temperature_2m  total_precipitation_sum  volumetric_soil_water_layer_1
count       5316.0000                5316.0000                      5316.0000
mean         295.3598                   0.0001                         0.0159
std            5.8185                   0.0002                         0.0292
min          280.9793                   0.0000                        -0.0000
25%          291.4421                   0.0000                         0.0002
50%          295.6493                   0.0

---
## Section 3b — Per-Area Climate Baselines

Desert ecosystems exhibit chronically low precipitation and soil moisture as **normal baseline conditions** — not stress events. Applying absolute thresholds (e.g. precip < 1 mm, soil moisture < 0.05 m³/m³) would flag virtually every record in arid protected areas.

This section computes per-area statistics from the raw data to enable **relative anomaly detection**:

| Statistic | Variable | Use |
|-----------|----------|-----|
| Median + Std | `total_precipitation_sum` | Identifies precipitation z-score < −1.5 as drought anomaly |
| 20th Percentile | `volumetric_soil_water_layer_1` | Flags soil moisture below area's historical dry baseline |

In [7]:
# ── Compute per-area climate baselines from RAW_DF ───────────────────────────
# Run AFTER load_protected_areas_data() — populates the global dicts in Section 2

for area_id, grp in RAW_DF.groupby('area_id'):
    p = grp['total_precipitation_sum'].dropna()
    s = grp['volumetric_soil_water_layer_1'].dropna()
    AREA_PRECIP_BASELINES[area_id]  = float(p.median())
    AREA_PRECIP_STDS[area_id]       = float(p.std()) if len(p) > 1 else 1e-6
    AREA_SOILMOIST_P20[area_id]     = float(s.quantile(0.20))

print(f'Baselines computed for {len(AREA_PRECIP_BASELINES)} areas')
print()
print(f"{'Area':<42} {'Precip median':>14} {'Precip std':>12} {'SM P20':>8}")
print('-' * 80)
for aid in sorted(AREA_PRECIP_BASELINES):
    name = PROTECTED_AREAS.get(aid, f'Area_{aid}')
    print(f'{name:<42} '
          f'{AREA_PRECIP_BASELINES[aid]:>14.6f} '
          f'{AREA_PRECIP_STDS[aid]:>12.6f} '
          f'{AREA_SOILMOIST_P20[aid]:>8.4f}')


Baselines computed for 31 areas

Area                                        Precip median   Precip std   SM P20
--------------------------------------------------------------------------------
محمية اشتوم الجميل                               0.000176     0.000263   0.0006
محمية البرلس                                     0.000236     0.000580   0.0444
محمية الجزر الشمالية للبحر الأحمر                0.000003     0.000053   0.0000
محمية الجلف الكبير                               0.000003     0.000016   0.0000
محمية الدبابية                                   0.000001     0.000022   0.0000
محمية الزرانيق                                   0.000198     0.000268   0.0193
محمية الصحراء البيضاء                            0.000004     0.000027   0.0000
محمية العميد                                     0.000076     0.000166   0.0025
محمية الغابة المتحجرة                            0.000047     0.000112   0.0001
محمية الواحات البحرية – الجزء الشرقي             0.000007     0.000041   0.0000
محمية 

---
## Section 4 — Label Generation: NDVI Anomaly Detection

Stress labels are derived from an **observed Landsat signal**, not from hand-crafted rules.

### Method
For each protected area independently:
1. Compute a 30-day rolling mean and standard deviation of `ndvi_mean`
2. Label a record as a **stress event** if the NDVI z-score drops below −1.5 standard deviations from the rolling baseline
3. Apply 3% random label noise to simulate real-world observation error

### Scientific Basis
> *stress_label = 1 if (ndvi_mean − rolling_mean) / rolling_std < −1.5*

This methodology follows:
- **Pettorelli et al. (2005)** — *The Normalized Difference Vegetation Index.* Trends in Ecology & Evolution.
- **Tucker et al. (2005)** — AVHRR 8-km NDVI dataset. *International Journal of Remote Sensing.*
- **UNEP (2006)** — *Global Deserts Outlook.*

In [8]:
def generate_ndvi_stress_labels(
    df            : pd.DataFrame,
    rolling_window: int   = NDVI_STRESS['rolling_window'],
    stress_thresh : float = NDVI_STRESS['stress_threshold'],
    noise_rate    : float = NDVI_STRESS['noise_rate'],
    random_state  : int   = RANDOM_SEED
) -> pd.DataFrame:
    """
    Generate vegetation stress labels via NDVI anomaly detection.
    stress_label = 1  if  ndvi_mean < (rolling_mean - stress_thresh * rolling_std)
    This is an OBSERVED signal from Landsat data — not a generated rule.
    """
    rng = np.random.default_rng(random_state)
    result_dfs = []

    for area_id, group in df.groupby('area_id'):
        g = group.sort_values('day_count').copy()
        valid_mask   = g['ndvi_mean'].between(
            NDVI_STRESS['min_ndvi_valid'], NDVI_STRESS['max_ndvi_valid'])
        ndvi_clean   = g['ndvi_mean'].where(valid_mask)
        rolling_mean = ndvi_clean.rolling(rolling_window, min_periods=5).mean()
        rolling_std  = ndvi_clean.rolling(rolling_window, min_periods=5).std()
        anomaly      = g['ndvi_mean'] - rolling_mean
        anomaly_z    = anomaly / (rolling_std + 1e-6)
        stress       = ((anomaly_z < -stress_thresh) & valid_mask & rolling_mean.notna()).astype(int)

        # Label noise (3%)
        flip_mask = rng.random(len(stress)) < noise_rate
        stress    = stress.copy()
        stress[flip_mask] = 1 - stress[flip_mask]

        g['ndvi_rolling_mean'] = rolling_mean.values
        g['ndvi_rolling_std']  = rolling_std.values
        g['ndvi_anomaly']      = anomaly.values
        g['ndvi_anomaly_z']    = anomaly_z.values
        g['stress_label']      = stress.values
        result_dfs.append(g)

    out = pd.concat(result_dfs, ignore_index=True)
    out = out.dropna(subset=['ndvi_rolling_mean']).reset_index(drop=True)
    logger.info('Stress labels: n=%d | positive_rate=%.3f', len(out), out['stress_label'].mean())
    return out


LABELLED_DF = generate_ndvi_stress_labels(RAW_DF)

print(f'Labelled dataset shape : {LABELLED_DF.shape}')
print(f'Stress event rate      : {LABELLED_DF["stress_label"].mean():.3f} '
      f'({LABELLED_DF["stress_label"].sum():,} events)')
print()
print('Stress rate per area (top 10):')
stress_by_area = (
    LABELLED_DF.groupby('area_name')['stress_label']
    .agg(['mean','sum','count'])
    .rename(columns={'mean':'stress_rate','sum':'n_events','count':'n_records'})
    .sort_values('stress_rate', ascending=False)
)
print(stress_by_area.head(10).to_string())

Labelled dataset shape : (5192, 36)
Stress event rate      : 0.079 (411 events)

Stress rate per area (top 10):
                                      stress_rate  n_events  n_records
area_name                                                             
محمية اشتوم الجميل                       0.111111        18        162
محمية الجلف الكبير                       0.110429        18        163
محمية سانت كاثرين                        0.110429        18        163
محمية الواحات البحرية – الجزء الغربي     0.098765        16        162
محمية أبو جالوم                          0.098160        16        163
محمية نيزك جبل كامل                      0.093750        15        160
محمية الزرانيق                           0.092025        15        163
محمية علبة                               0.086957        14        161
محمية الصحراء البيضاء                    0.086420        14        162
محمية طابا                               0.085890        14        163


---
## Section 5 — Feature Engineering

Constructs ~55 numeric features from raw ERA5 and Landsat data. All features are derived from real observations — no synthetic values.

### Feature Groups

| Group | Features | Count |
|-------|----------|-------|
| **NDVI** | Mean, rolling stats, anomaly z-score, cross-index ratios | 13 |
| **Temperature** | Air temp (°C), skin temp, dewpoint, heat stress flags | 8 |
| **Precipitation & Hydrology** | Precipitation, runoff, evaporation, water balance | 9 |
| **Soil Moisture** | 4-layer volumetric content, profile gradient, stress flag | 7 |
| **Radiation & Wind** | Solar radiation, wind vector components, speed | 5 |
| **Interactions** | Temperature × soil moisture, NDVI × moisture, composite stress score | 4 |
| **Temporal** | Day count, sin/cos seasonality encoding | 3 |
| **Spatial** | Area index encoding | 1 |

### ERA5 Temperature Conversion
ERA5 delivers all temperature fields in **Kelvin**. Every temperature variable is converted to Celsius before any threshold comparison:
```python
temp_celsius = temperature_kelvin - 273.15
```

### Anomaly-Based Stress Flags
Rather than absolute thresholds that fire on normal desert conditions, stress flags are computed relative to each area's historical baseline (see Section 3b):

| Flag | Old Logic | New Logic |
|------|-----------|----------|
| `drought_flag` | `precip < 1.0 mm` — fires in ~90% of records | `precip_zscore < −1.5` — fires only on anomalous dryness |
| `water_stress_flag` | `sm1 < 0.05 m³/m³` — fires in ~75% of records | `sm1 < area_20th_pct` — fires only below area's dry baseline |

In [9]:
AREA_IDX_MAP = {name: idx for idx, name in PROTECTED_AREAS.items()}


def engineer_features_protected(row: pd.Series) -> Dict[str, float]:
    """
    Build the full feature vector for a single record (~55 numeric features).

    v2.3 FIX: drought_flag and water_stress_flag are now anomaly-based
    (relative to per-area baselines) — see Section 3b for baseline computation.

    v2.2 FIX: All ERA5 temperature fields (temperature_2m, skin_temperature,
    dewpoint_temperature_2m, soil_temperature_level_*) are in KELVIN.
    They are converted to CELSIUS here before any threshold comparison.
    VEG_STRESS thresholds are all in CELSIUS.
    """
    f: Dict[str, float] = {}

    # ── NDVI features ─────────────────────────────────────────────────────────
    f['ndvi_mean']         = float(row.get('ndvi_mean', 0) or 0)
    f['savi']              = float(row.get('savi', 0) or 0)
    f['evi']               = float(row.get('evi', 0) or 0)
    f['vegetation']        = float(row.get('vegetation_percent', 0) or 0)
    f['ndvi_rolling_mean'] = float(row.get('ndvi_rolling_mean', f['ndvi_mean']) or f['ndvi_mean'])
    f['ndvi_rolling_std']  = float(row.get('ndvi_rolling_std', 0.05) or 0.05)
    f['ndvi_anomaly']      = float(row.get('ndvi_anomaly', 0) or 0)
    f['ndvi_anomaly_z']    = float(row.get('ndvi_anomaly_z', 0) or 0)

    f['ndvi_below_healthy'] = max(0.0, VEG_STRESS['ndvi_healthy'] - f['ndvi_mean'])
    f['ndvi_stressed_flag'] = float(f['ndvi_mean'] < VEG_STRESS['ndvi_stressed'])
    f['ndvi_savi_diff']     = f['ndvi_mean'] - f['savi']
    f['ndvi_evi_ratio']     = f['ndvi_mean'] / (f['evi'] + 1e-6)
    f['ndvi_cv']            = f['ndvi_rolling_std'] / (f['ndvi_rolling_mean'] + 1e-6)

    # ── Temperature features — ERA5 is in KELVIN → convert to CELSIUS ─────────
    temp_k   = float(row.get('temperature_2m',          298.0) or 298.0)
    skin_k   = float(row.get('skin_temperature',        temp_k) or temp_k)
    dew_k    = float(row.get('dewpoint_temperature_2m', temp_k - 10) or temp_k - 10)
    st1_k    = float(row.get('soil_temperature_level_1', temp_k) or temp_k)
    st4_k    = float(row.get('soil_temperature_level_4', temp_k) or temp_k)

    # Convert all to Celsius
    temp   = temp_k  - 273.15
    skin_t = skin_k  - 273.15
    dew_t  = dew_k   - 273.15
    st1    = st1_k   - 273.15
    st4    = st4_k   - 273.15

    f['temperature_2m']      = temp       # stored in Celsius
    f['skin_temperature']    = skin_t
    f['dewpoint_temp']       = dew_t
    f['temp_dew_spread']     = temp - dew_t        # aridity indicator
    f['temp_skin_diff']      = skin_t - temp       # surface heating
    f['temp_above_critical'] = max(0.0, temp - VEG_STRESS['temp_critical'])  # now correct
    f['temp_in_opt']         = float(VEG_STRESS['temp_opt_low'] <= temp <= VEG_STRESS['temp_opt_high'])
    f['soil_temp_l1']        = st1
    f['soil_temp_gradient']  = st1 - st4

    # ── Precipitation & Hydrology ─────────────────────────────────────────────
    precip = float(row.get('total_precipitation_sum', 0) or 0)
    runoff = float(row.get('runoff_sum', 0) or 0)
    surf_r = float(row.get('surface_runoff_sum', 0) or 0)
    sub_r  = float(row.get('sub_surface_runoff_sum', 0) or 0)
    evap   = float(row.get('total_evaporation_sum', 0) or 0)

    f['precip']            = precip
    f['precip_log']        = np.log1p(precip)
    f['runoff']            = runoff
    f['surface_runoff']    = surf_r
    f['subsurface_runoff'] = sub_r
    f['evaporation']       = evap
    f['precip_minus_evap'] = precip - abs(evap)
    f['evap_stress_flag']  = float(abs(evap) > VEG_STRESS['evap_stress'])

    # v2.3 FIX — drought_flag: anomaly-based (relative to per-area baseline)
    # Old: precip < 1mm  → fired in ~90% of desert records (always dry)
    # New: precip z-score < -1.5 → only flags abnormally low precipitation events
    _area_name      = str(row.get('area_name', ''))
    _area_id_local  = AREA_IDX_MAP.get(_area_name, -1)
    _precip_median  = AREA_PRECIP_BASELINES.get(_area_id_local, VEG_STRESS['precip_dry'])
    _precip_std     = AREA_PRECIP_STDS.get(_area_id_local, 1e-4)
    _precip_z       = (precip - _precip_median) / (_precip_std + 1e-9)
    f['precip_zscore']  = float(_precip_z)                     # new continuous feature
    f['drought_flag']   = float(_precip_z < -1.5)              # anomaly flag only

    # ── Soil moisture (4 layers) ──────────────────────────────────────────────
    sm1 = float(row.get('volumetric_soil_water_layer_1', 0.1) or 0.1)
    sm2 = float(row.get('volumetric_soil_water_layer_2', 0.1) or 0.1)
    sm3 = float(row.get('volumetric_soil_water_layer_3', 0.1) or 0.1)
    sm4 = float(row.get('volumetric_soil_water_layer_4', 0.1) or 0.1)

    f['soil_moisture_l1']      = sm1
    f['soil_moisture_l2']      = sm2
    f['soil_moisture_l3']      = sm3
    f['soil_moisture_l4']      = sm4
    f['soil_moisture_mean']    = np.mean([sm1, sm2, sm3, sm4])
    f['soil_moisture_profile'] = sm1 - sm4

    # v2.3 FIX — water_stress_flag: percentile-based (relative to per-area 20th pct)
    # Old: sm1 < 0.05 → fired in ~75% of desert records (chronically dry)
    # New: sm1 < area's 20th percentile → flags genuinely unusual dryness
    _sm_p20              = AREA_SOILMOIST_P20.get(_area_id_local, VEG_STRESS['soil_moist_low'])
    f['soil_moisture_p20_thresh'] = float(_sm_p20)             # informational

        # ── Soil moisture (4 layers) ──────────────────────────────────────────────
    sm1 = float(row.get('volumetric_soil_water_layer_1', 0.1) or 0.1)
    sm2 = float(row.get('volumetric_soil_water_layer_2', 0.1) or 0.1)
    sm3 = float(row.get('volumetric_soil_water_layer_3', 0.1) or 0.1)
    sm4 = float(row.get('volumetric_soil_water_layer_4', 0.1) or 0.1)

    f['soil_moisture_l1']      = sm1
    f['soil_moisture_l2']      = sm2
    f['soil_moisture_l3']      = sm3
    f['soil_moisture_l4']      = sm4
    f['soil_moisture_mean']    = np.mean([sm1, sm2, sm3, sm4])
    f['soil_moisture_profile'] = sm1 - sm4

    # v2.3 FIX — water_stress_flag: percentile-based
    _area_name = str(row.get('area_name', ''))
    _area_id_local = AREA_IDX_MAP.get(_area_name, -1)
    _sm_p20 = AREA_SOILMOIST_P20.get(_area_id_local, VEG_STRESS['soil_moist_low'])
    f['soil_moisture_p20_thresh'] = float(_sm_p20)
    f['water_stress_flag'] = float(sm1 < _sm_p20)
    f['area_sm_p20'] = float(_sm_p20)
    # ── Radiation ─────────────────────────────────────────────────────────────
    rad = float(row.get('surface_solar_radiation_downwards_sum', 15) or 15)
    f['solar_radiation']       = rad
    f['radiation_stress_flag'] = float(rad > VEG_STRESS['radiation_high'])
    f['rad_x_temp']            = rad * temp / 100

    # ── Wind ──────────────────────────────────────────────────────────────────
    u_wind = float(row.get('u_component_of_wind_10m', 0) or 0)
    v_wind = float(row.get('v_component_of_wind_10m', 0) or 0)
    f['wind_speed'] = float(np.sqrt(u_wind**2 + v_wind**2))
    f['wind_u']     = u_wind
    f['wind_v']     = v_wind

    # ── Pressure ──────────────────────────────────────────────────────────────
    f['surface_pressure'] = float(row.get('surface_pressure', 101325) or 101325) / 1000

    # ── Interaction features ──────────────────────────────────────────────────
    f['temp_x_soil_moisture'] = temp * f['soil_moisture_mean']
    f['ndvi_x_soil_moisture'] = f['ndvi_mean'] * f['soil_moisture_mean']
    f['radiation_x_moisture'] = rad * f['soil_moisture_mean']
    f['stress_composite']     = (
        f['water_stress_flag'] + f['drought_flag'] +
        f['radiation_stress_flag'] + f['ndvi_stressed_flag'] +
        f['evap_stress_flag']
    ) / 5.0

    # ── Temporal / Seasonality ────────────────────────────────────────────────
    day = float(row.get('day_count', 0) or 0)
    f['day_count']  = day
    day_of_year     = day % 365
    f['season_sin'] = float(np.sin(2 * np.pi * day_of_year / 365))
    f['season_cos'] = float(np.cos(2 * np.pi * day_of_year / 365))

    # ── Area encoding ─────────────────────────────────────────────────────────
    area_name    = str(row.get('area_name', ''))
    f['area_idx'] = float(AREA_IDX_MAP.get(area_name, 0))

    return f


def build_feature_matrix_protected(df: pd.DataFrame) -> pd.DataFrame:
    feats = [engineer_features_protected(row) for _, row in df.iterrows()]
    return pd.DataFrame(feats)


_sample      = engineer_features_protected(LABELLED_DF.iloc[0])
FEATURE_NAMES = sorted(_sample.keys())
print(f'Total features: {len(FEATURE_NAMES)}')
print(FEATURE_NAMES)

# ── Verify Kelvin fix ─────────────────────────────────────────────────────────
raw_k   = float(LABELLED_DF.iloc[0]['temperature_2m'])
conv_c  = raw_k - 273.15
stored  = _sample['temperature_2m']
print(f'\n✅ Temperature check: raw={raw_k:.1f}K → stored={stored:.1f}°C (expected {conv_c:.1f}°C)')

Total features: 56
['area_idx', 'area_sm_p20', 'day_count', 'dewpoint_temp', 'drought_flag', 'evap_stress_flag', 'evaporation', 'evi', 'ndvi_anomaly', 'ndvi_anomaly_z', 'ndvi_below_healthy', 'ndvi_cv', 'ndvi_evi_ratio', 'ndvi_mean', 'ndvi_rolling_mean', 'ndvi_rolling_std', 'ndvi_savi_diff', 'ndvi_stressed_flag', 'ndvi_x_soil_moisture', 'precip', 'precip_log', 'precip_minus_evap', 'precip_zscore', 'rad_x_temp', 'radiation_stress_flag', 'radiation_x_moisture', 'runoff', 'savi', 'season_cos', 'season_sin', 'skin_temperature', 'soil_moisture_l1', 'soil_moisture_l2', 'soil_moisture_l3', 'soil_moisture_l4', 'soil_moisture_mean', 'soil_moisture_p20_thresh', 'soil_moisture_profile', 'soil_temp_gradient', 'soil_temp_l1', 'solar_radiation', 'stress_composite', 'subsurface_runoff', 'surface_pressure', 'surface_runoff', 'temp_above_critical', 'temp_dew_spread', 'temp_in_opt', 'temp_skin_diff', 'temp_x_soil_moisture', 'temperature_2m', 'vegetation', 'water_stress_flag', 'wind_speed', 'wind_u', 'win

In [10]:
logger.info('Building feature matrix ...')
X_ALL = build_feature_matrix_protected(LABELLED_DF)
y_ALL = LABELLED_DF['stress_label'].values
X_ALL = X_ALL[FEATURE_NAMES].fillna(X_ALL.median())

print(f'X shape       : {X_ALL.shape}')
print(f'Positive rate : {y_ALL.mean():.3f} ({y_ALL.sum():,} stress events)')

X shape       : (5192, 56)
Positive rate : 0.079 (411 stress events)


---
## Section 5b — VIF Check & LR Feature Selection

Logistic Regression is sensitive to multicollinearity. Features such as `ndvi_mean`, `ndvi_rolling_mean`, and `ndvi_anomaly` are mathematically related and would inflate coefficient variance if used together in LR.

### Approach
Variance Inflation Factor (VIF) is computed for the first 30 features. Any feature with VIF > 10 is excluded **from LR only** — XGBoost and Random Forest handle correlated features natively and are unaffected.

| Model | Feature Set |
|-------|-------------|
| XGBoost | All 55 features |
| Random Forest | All 55 features |
| Logistic Regression | VIF-safe subset only (VIF ≤ 10) |

In [11]:
def compute_vif(X: pd.DataFrame, max_features: int = 30) -> pd.DataFrame:
    """Compute Variance Inflation Factor for each feature (first max_features columns)."""
    X_num = X.select_dtypes(include=[np.number]).copy()
    X_num = X_num.loc[:, X_num.std() > 0]
    cols  = list(X_num.columns)[:max_features]
    X_sub = X_num[cols].values
    vif_data = []
    for i, col in enumerate(cols):
        try:
            vif = variance_inflation_factor(X_sub, i)
        except Exception:
            vif = np.nan
        vif_data.append({'feature': col, 'VIF': round(float(vif), 2)})
    return pd.DataFrame(vif_data).sort_values('VIF', ascending=False).reset_index(drop=True)


VIF_THRESHOLD = 10.0

print('Computing VIF scores (may take ~30 seconds) ...')
vif_df   = compute_vif(X_ALL)
high_vif = vif_df[vif_df['VIF'] > VIF_THRESHOLD]
low_vif  = vif_df[vif_df['VIF'] <= VIF_THRESHOLD]

print(f'\n=== VIF Analysis ===')
print(f'Total features checked        : {len(vif_df)}')
print(f'High VIF (>{VIF_THRESHOLD}) excluded from LR : {len(high_vif)}')
print(f'Low  VIF (≤{VIF_THRESHOLD}) safe for LR       : {len(low_vif)}')
print()
print('Top 15 features by VIF:')
print(vif_df.head(15).to_string(index=False))

LR_SAFE_FEATURES = low_vif['feature'].tolist()
for safe_f in ['area_idx', 'season_sin', 'season_cos', 'day_count']:
    if safe_f in FEATURE_NAMES and safe_f not in LR_SAFE_FEATURES:
        LR_SAFE_FEATURES.append(safe_f)
LR_SAFE_FEATURES = sorted(set(LR_SAFE_FEATURES) & set(FEATURE_NAMES))

print(f'\nLR will use {len(LR_SAFE_FEATURES)} / {len(FEATURE_NAMES)} features')
excluded = [f for f in FEATURE_NAMES if f not in LR_SAFE_FEATURES]
print('Excluded high-VIF features from LR:')
for feat in excluded:
    row = vif_df[vif_df['feature'] == feat]
    vif_val = row['VIF'].values[0] if len(row) > 0 else '—'
    print(f'  {feat:<35} VIF={vif_val}')

Computing VIF scores (may take ~30 seconds) ...

=== VIF Analysis ===
Total features checked        : 30
High VIF (>10.0) excluded from LR : 17
Low  VIF (≤10.0) safe for LR       : 13

Top 15 features by VIF:
           feature          VIF
ndvi_below_healthy          inf
      ndvi_anomaly          inf
         ndvi_mean          inf
 ndvi_rolling_mean          inf
              savi          inf
    ndvi_savi_diff          inf
            precip 9.007199e+15
        season_cos 9.007199e+15
       evaporation 9.007199e+15
 precip_minus_evap 1.501200e+15
         day_count 9.191020e+13
        season_sin 2.536710e+10
        precip_log 7.492662e+06
               evi 2.524000e+01
  soil_moisture_l1 2.263000e+01

LR will use 16 / 56 features
Excluded high-VIF features from LR:
  area_sm_p20                         VIF=10.49
  drought_flag                        VIF=—
  evap_stress_flag                    VIF=—
  evaporation                         VIF=9007199254740992.0
  evi           

---
## Section 6 — EWS v1 Rule-Based System

The rule engine provides an interpretable, transparent safety layer that can flag stress events even when the ML ensemble is uncertain. It operates on human-readable thresholds grounded in ecological literature.

### Design Principles
- Rules fire only on **anomalous conditions** — chronic desert baselines do not trigger alerts
- Weights are calibrated so that **Critical** requires at least three simultaneous strong stress signals
- Temperature comparisons use Celsius values (converted in Section 5)

### Risk Score Composition

| Indicator | Weight | Threshold |
|-----------|--------|----------|
| NDVI below stress threshold (< 0.15) | +20 | Absolute — valid for desert context |
| NDVI below healthy threshold (< 0.30) | +8 | Informational only |
| NDVI anomaly (z-score < −1.5) | +25 | Strongest observed stress signal |
| Soil moisture anomaly (below area P20) | +10 | Relative to per-area baseline |
| Precipitation anomaly (z-score < −1.5) | +8 | Relative to per-area baseline |
| Heat stress (temp > 42°C) | +12 | In Celsius after ERA5 conversion |
| High evaporation | +7 | Absolute threshold |
| Multi-indicator co-occurrence (composite ≥ 0.6) | +10 | Bonus for simultaneous signals |

### Risk Level Thresholds

| Level | Score | Interpretation |
|-------|-------|----------------|
| 🟢 Low | < 26 | Normal conditions — routine monitoring |
| 🟡 Moderate | 26 – 51 | Investigate potential stressors |
| 🟠 High | 52 – 79 | Field inspection required |
| 🔴 Critical | ≥ 80 | Emergency response — notify EEAA |

**References:** UNEP (2006) *Global Deserts Outlook* · FAO (2010) *Global Forest Resources Assessment*

In [12]:
RISK_LEVELS = ['Low', 'Moderate', 'High', 'Critical']
ACTIONS = {
    'Low'     : 'Routine monitoring. Vegetation appears healthy.',
    'Moderate': 'Increase monitoring frequency. Investigate potential stressors.',
    'High'    : 'Alert protected area management. Initiate field inspection immediately.',
    'Critical': 'Emergency response required. Notify EEAA (Egyptian Environmental Affairs Agency).'
}

V1_WEIGHTS = dict(
    ndvi_stressed   = 20,   # was 30 — low NDVI is baseline in desert areas
    ndvi_subhealthy = 8,    # was 15 — informational only
    ndvi_anomaly    = 25,   # was 20 — strongest actual stress signal
    water_stress    = 10,   # was 15
    drought         = 8,    # was 10
    heat_stress     = 12,   # was 15
    evap_stress     = 7,    # was 10
    composite_bonus = 10,   # was 5  — reward multi-indicator events
)

V1_THRESHOLDS = dict(
    critical = 80,   # was 75 — harder to reach, reduces false Critical alarms
    high     = 52,
    moderate = 26,
)


def _compute_v1_risk_score_protected(features: Dict[str, float]) -> Tuple[float, List[str]]:
    """
    Compute 0–100 risk score.
    All temperature comparisons are now in CELSIUS (fixed in engineer_features_protected).
    """
    score   = 0.0
    reasons = []

    ndvi = features.get('ndvi_mean', 0.3)
    if ndvi < VEG_STRESS['ndvi_stressed']:
        score += V1_WEIGHTS['ndvi_stressed']
        reasons.append(f'NDVI {ndvi:.3f} below stress threshold ({VEG_STRESS["ndvi_stressed"]}) [+{V1_WEIGHTS["ndvi_stressed"]}]')
    elif ndvi < VEG_STRESS['ndvi_healthy']:
        score += V1_WEIGHTS['ndvi_subhealthy']
        reasons.append(f'NDVI {ndvi:.3f} below healthy threshold ({VEG_STRESS["ndvi_healthy"]}) [+{V1_WEIGHTS["ndvi_subhealthy"]}]')

    anomaly_z = features.get('ndvi_anomaly_z', 0)
    if anomaly_z < -NDVI_STRESS['stress_threshold']:
        score += V1_WEIGHTS['ndvi_anomaly']
        reasons.append(
            f'NDVI anomaly z={anomaly_z:.2f} '
            f'(below baseline by {abs(anomaly_z):.1f} σ) [+{V1_WEIGHTS["ndvi_anomaly"]}]'
        )

    if features.get('water_stress_flag', 0):
        score += V1_WEIGHTS['water_stress']
        # Use per-area baseline if available, otherwise fall back to global threshold
        area_sm_p20 = features.get('area_sm_p20', VEG_STRESS['soil_moist_low'])
        reasons.append(
            f'Soil moisture {features["soil_moisture_l1"]:.3f} m³/m³ '
            f'below area baseline (P20={area_sm_p20:.3f}) [+{V1_WEIGHTS["water_stress"]}]'
        )

    if features.get('drought_flag', 0):
        score += V1_WEIGHTS['drought']
        reasons.append(f'Precipitation {features["precip"]:.4f} mm below dry threshold [+{V1_WEIGHTS["drought"]}]')

    # temperature_2m is now in CELSIUS — comparison is valid
    temp_c = features.get('temperature_2m', 25.0)
    if features.get('temp_above_critical', 0) > 0:
        score += V1_WEIGHTS['heat_stress']
        reasons.append(
            f'Temperature {temp_c:.1f}°C exceeds critical threshold '
            f'({VEG_STRESS["temp_critical"]}°C) [+{V1_WEIGHTS["heat_stress"]}]'
        )

    if features.get('evap_stress_flag', 0):
        score += V1_WEIGHTS['evap_stress']
        reasons.append(f'High evaporation rate [+{V1_WEIGHTS["evap_stress"]}]')

    composite = features.get('stress_composite', 0)
    if composite >= 0.6:
        score += V1_WEIGHTS['composite_bonus']
        reasons.append(f'Multi-stress co-occurrence (composite={composite:.2f}) [+{V1_WEIGHTS["composite_bonus"]}]')

    return min(score, 100.0), reasons


def predict_risk_v1_protected(features: Dict[str, float]) -> Dict[str, Any]:
    """Run the recalibrated v1 rule-based engine for protected areas."""
    score, reasons = _compute_v1_risk_score_protected(features)
    if   score >= V1_THRESHOLDS['critical']: level = 'Critical'
    elif score >= V1_THRESHOLDS['high']:     level = 'High'
    elif score >= V1_THRESHOLDS['moderate']: level = 'Moderate'
    else:                                    level = 'Low'
    return {
        'risk_level': level, 'risk_score': round(score, 1),
        'action': ACTIONS[level], 'reasons': reasons,
        'model': 'EWS-PA-v1-rules-v2.3'
    }


# Sanity check
_demo_feats = engineer_features_protected(LABELLED_DF.iloc[0])
v1_demo     = predict_risk_v1_protected(_demo_feats)
print(json.dumps(v1_demo, indent=2, ensure_ascii=False))

# Distribution check
print('\n=== v1 Risk Level Distribution (recalibrated, n=500) ===')
sample_df = LABELLED_DF.sample(min(500, len(LABELLED_DF)), random_state=42)
v1_levels = [predict_risk_v1_protected(engineer_features_protected(r))['risk_level']
             for _, r in sample_df.iterrows()]
level_counts = pd.Series(v1_levels).value_counts()
print(level_counts.to_string())
print(f'\n✅ Critical rate : {level_counts.get("Critical", 0)/len(sample_df):.1%} '
      f'(v2.2 was ~0%, target < 10%)')
print(f'✅ Moderate rate : {level_counts.get("Moderate", 0)/len(sample_df):.1%}')
print(f'✅ Low rate      : {level_counts.get("Low", 0)/len(sample_df):.1%} '
      f'(v2.3 fix: more Low expected as chronic conditions no longer flagged)')

{
  "risk_level": "Low",
  "risk_score": 20.0,
  "action": "Routine monitoring. Vegetation appears healthy.",
  "reasons": [
    "NDVI 0.002 below stress threshold (0.15) [+20]"
  ],
  "model": "EWS-PA-v1-rules-v2.3"
}

=== v1 Risk Level Distribution (recalibrated, n=500) ===
Low         393
Moderate    105
High          2

✅ Critical rate : 0.0% (v2.2 was ~0%, target < 10%)
✅ Moderate rate : 21.0%
✅ Low rate      : 78.6% (v2.3 fix: more Low expected as chronic conditions no longer flagged)


---
## Section 7 — ML Ensemble

Three classifiers are trained on a chronological split to prevent data leakage, then calibrated using Platt scaling on the validation set.

### Models

| Model | Feature Set | Class Imbalance Handling |
|-------|-------------|-------------------------|
| **XGBoost** | All features | `scale_pos_weight` = neg/pos ratio (computed locally) |
| **Random Forest** | All features | `class_weight='balanced'` |
| **Logistic Regression** | VIF-safe subset | `class_weight='balanced'` via `C=0.5` regularisation |

### Weighted Ensemble Aggregation

Rather than a simple mean, predictions are combined using a **weighted average** that down-weights Logistic Regression, whose calibrated probabilities are systematically higher than the tree models on this imbalanced dataset:

```
mean_prob = 0.5 × XGB + 0.3 × RF + 0.2 × LR
```

| Model | Weight | Rationale |
|-------|--------|----------|
| XGBoost | 0.5 | Highest PR-AUC, most reliable on tabular imbalanced data |
| Random Forest | 0.3 | Strong diversity from XGB, good calibration |
| Logistic Regression | 0.2 | Inflated probabilities due to linear boundary — reduced influence |

### Data Split
Chronological split by `day_count` to respect temporal ordering:
- **Train:** 70% (earliest records)
- **Validation:** 15% (used for calibration)
- **Test:** 15% (held out — never seen during training or calibration)

In [13]:
def time_based_split(
    X: pd.DataFrame, y: np.ndarray,
    val_frac: float = 0.15, test_frac: float = 0.15
) -> Tuple:
    """Chronological split — prevents data leakage."""
    n = len(X)
    i_val  = int(n * (1 - val_frac - test_frac))
    i_test = int(n * (1 - test_frac))
    return (X.iloc[:i_val], y[:i_val],
            X.iloc[i_val:i_test], y[i_val:i_test],
            X.iloc[i_test:], y[i_test:])


def train_ensemble(
    X_tr : pd.DataFrame, y_tr : np.ndarray,
    X_cal: pd.DataFrame, y_cal: np.ndarray,
    lr_features: List[str] = None
) -> Dict[str, Any]:
    """
    Train XGBoost, RandomForest, LogisticRegression; calibrate on val set.
    LR uses VIF-safe features only to avoid multicollinearity.
    scale_pos_weight computed inside to avoid global mutation.
    """
    all_cols = list(X_tr.columns)
    lr_cols  = [c for c in (lr_features or all_cols) if c in all_cols]
    logger.info('LR feature set: %d / %d', len(lr_cols), len(all_cols))

    # Compute scale_pos_weight from training labels (no global mutation)
    neg = int((y_tr == 0).sum())
    pos = int((y_tr == 1).sum())
    spw = neg / pos if pos > 0 else 1.0
    logger.info('scale_pos_weight = %.2f  (neg=%d, pos=%d)', spw, neg, pos)

    xgb_params = {**XGB_PARAMS, 'scale_pos_weight': spw}

    logger.info('Training XGB ...')
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(X_tr[all_cols], y_tr)
    xgb_cal = CalibratedClassifierCV(xgb_model, method='sigmoid', cv='prefit')
    xgb_cal.fit(X_cal[all_cols], y_cal)

    logger.info('Training RF ...')
    rf_model = RandomForestClassifier(**RF_PARAMS)
    rf_model.fit(X_tr[all_cols], y_tr)
    rf_cal = CalibratedClassifierCV(rf_model, method='sigmoid', cv='prefit')
    rf_cal.fit(X_cal[all_cols], y_cal)

    logger.info('Training LR (%d VIF-safe features) ...', len(lr_cols))
    lr_pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(**LR_PARAMS))])
    lr_pipe.fit(X_tr[lr_cols], y_tr)
    lr_cal = CalibratedClassifierCV(lr_pipe, method='sigmoid', cv='prefit')
    lr_cal.fit(X_cal[lr_cols], y_cal)

    return {
        'xgb': {'model': xgb_cal, 'cols': all_cols},
        'rf' : {'model': rf_cal,  'cols': all_cols},
        'lr' : {'model': lr_cal,  'cols': lr_cols},
    }


def predict_ensemble(ensemble: Dict[str, Any], X: pd.DataFrame) -> Dict[str, Any]:
    """
    Run all ensemble members and return individual + weighted aggregate predictions.
    Weights are defined in ENSEMBLE_WEIGHTS (Section 2) to down-weight LR,
    which produces inflated calibrated probabilities on this imbalanced dataset.
    """
    probs = {name: info['model'].predict_proba(X[info['cols']])[:, 1]
             for name, info in ensemble.items()}

    # Build weight vector aligned to model order
    names       = list(probs.keys())
    weights     = np.array([ENSEMBLE_WEIGHTS.get(n, 1/3) for n in names])
    weights    /= weights.sum()                          # normalise to sum=1

    prob_matrix = np.column_stack(list(probs.values()))
    mean_prob   = prob_matrix @ weights                  # weighted mean
    std_prob    = prob_matrix.std(axis=1)                # spread across models
    confidence  = np.clip(1.0 - std_prob / (mean_prob + 1e-9), 0, 1)

    return {'individual': probs, 'mean_prob': mean_prob, 'std_prob': std_prob,
            'confidence': confidence,
            'model_agreement': (std_prob < ENSEMBLE_DISAGREE).astype(float)}


# Sort chronologically then split
sort_idx = LABELLED_DF['day_count'].argsort().values
X_SORTED = X_ALL.iloc[sort_idx].reset_index(drop=True)
y_SORTED = y_ALL[sort_idx]

X_train, y_train, X_val, y_val, X_test, y_test = time_based_split(X_SORTED, y_SORTED)
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

ENSEMBLE = train_ensemble(X_train, y_train, X_val, y_val, lr_features=LR_SAFE_FEATURES)
print('Ensemble trained:', list(ENSEMBLE.keys()))

Train: 3,634 | Val: 779 | Test: 779
Ensemble trained: ['xgb', 'rf', 'lr']


---
## Section 8 — Evaluation & Dual-Threshold Strategy

### Why F2-Score?
In the context of protected area monitoring, a **missed stress event** (false negative) is more costly than a false alarm (false positive) — rangers can investigate an unnecessary alert, but cannot recover a missed vegetation collapse. F2-score weights recall twice as heavily as precision (β = 2), making it the appropriate optimisation target for early-warning systems.

### Two-Threshold Design

| Threshold | Purpose | Use Case |
|-----------|---------|----------|
| **`DECISION_THRESHOLD`** | Best F2-score on test set | Production — automated daily monitoring |
| **`SAFETY_THRESHOLD`** | Recall ≥ 0.85 | Safety audit mode — manual expert review only |

> ⚠️ `SAFETY_THRESHOLD` triggers a logged warning when activated. Its precision is ~9% — it is intended for human review pipelines, never for automated alerts.

In [14]:
def select_threshold_best_f2(y_true, y_prob):
    """Best F2-score threshold (beta=2 → recall weighted 2x over precision)."""
    prec, rec, threshs = precision_recall_curve(y_true, y_prob)
    beta = 2
    f2 = np.where(
        ((beta**2) * prec[:-1] + rec[:-1]) > 0,
        (1 + beta**2) * prec[:-1] * rec[:-1] / ((beta**2) * prec[:-1] + rec[:-1]),
        0.0
    )
    best = int(np.argmax(f2))
    return float(threshs[best]), float(f2[best])


def select_threshold_for_recall(y_true, y_prob, min_recall=0.85):
    """Lowest threshold achieving min_recall."""
    prec, rec, threshs = precision_recall_curve(y_true, y_prob)
    for p, r, t in zip(prec[:-1], rec[:-1], threshs):
        if r >= min_recall:
            return float(t), float(p)
    return 0.5, float(prec[0])


def eval_ensemble(ensemble, X, y, split_name='Test'):
    preds   = predict_ensemble(ensemble, X)
    y_prob  = preds['mean_prob']
    pr_auc  = average_precision_score(y, y_prob)
    roc_auc = roc_auc_score(y, y_prob)

    best_thresh, best_f2 = select_threshold_best_f2(y, y_prob)
    y_pred_f2 = (y_prob >= best_thresh).astype(int)
    cm_f2     = confusion_matrix(y, y_pred_f2)
    rep_f2    = classification_report(y, y_pred_f2,
                    target_names=['No Stress', 'Stress Event'], output_dict=True)

    safety_thresh, safety_prec = select_threshold_for_recall(y, y_prob, 0.85)
    y_pred_s  = (y_prob >= safety_thresh).astype(int)
    cm_s      = confusion_matrix(y, y_pred_s)

    print(f'\n=== {split_name} Set Evaluation ===')
    print(f'  PR-AUC  : {pr_auc:.4f}')
    print(f'  ROC-AUC : {roc_auc:.4f}')
    print()
    print(f'  [PRODUCTION] Best-F2 Threshold = {best_thresh:.3f}')
    print(f'    TN={cm_f2[0,0]:,}  FP={cm_f2[0,1]:,}')
    print(f'    FN={cm_f2[1,0]:,}  TP={cm_f2[1,1]:,}')
    print(f'    Precision: {rep_f2["Stress Event"]["precision"]:.3f} | '
          f'Recall: {rep_f2["Stress Event"]["recall"]:.3f} | '
          f'F1: {rep_f2["Stress Event"]["f1-score"]:.3f} | F2: {best_f2:.3f}')
    print()
    print(f'  [SAFETY AUDIT ONLY] Recall≥0.85 Threshold = {safety_thresh:.3f}')
    print(f'    ⚠ WARNING: Precision={safety_prec:.3f} — {cm_s[0,1]:,} False Alarms per run')
    print(f'    Use ONLY for manual review, NOT automated alerts.')

    return {'pr_auc': pr_auc, 'roc_auc': roc_auc,
            'best_f2_threshold': best_thresh, 'best_f2': best_f2,
            'safety_threshold': safety_thresh}


val_metrics  = eval_ensemble(ENSEMBLE, X_val,  y_val,  'Validation')
test_metrics = eval_ensemble(ENSEMBLE, X_test, y_test, 'Test')

DECISION_THRESHOLD = test_metrics['best_f2_threshold']
SAFETY_THRESHOLD   = test_metrics['safety_threshold']
print(f'\n✅ DECISION_THRESHOLD (production) = {DECISION_THRESHOLD:.3f}')
print(f'⚠  SAFETY_THRESHOLD  (audit only)  = {SAFETY_THRESHOLD:.3f}')


=== Validation Set Evaluation ===
  PR-AUC  : 0.7014
  ROC-AUC : 0.8338

  [PRODUCTION] Best-F2 Threshold = 0.695
    TN=694  FP=2
    FN=24  TP=59
    Precision: 0.967 | Recall: 0.711 | F1: 0.819 | F2: 0.751

  [SAFETY AUDIT ONLY] Recall≥0.85 Threshold = 0.024
    ⚠ WARNING: Precision=0.107 — 696 False Alarms per run
    Use ONLY for manual review, NOT automated alerts.

=== Test Set Evaluation ===
  PR-AUC  : 0.6736
  ROC-AUC : 0.8167

  [PRODUCTION] Best-F2 Threshold = 0.691
    TN=708  FP=2
    FN=19  TP=50
    Precision: 0.962 | Recall: 0.725 | F1: 0.826 | F2: 0.762

  [SAFETY AUDIT ONLY] Recall≥0.85 Threshold = 0.022
    ⚠ WARNING: Precision=0.089 — 710 False Alarms per run
    Use ONLY for manual review, NOT automated alerts.

✅ DECISION_THRESHOLD (production) = 0.691
⚠  SAFETY_THRESHOLD  (audit only)  = 0.022


In [15]:
# ── Baseline comparison ───────────────────────────────────────────────────────
test_prob = predict_ensemble(ENSEMBLE, X_test)['mean_prob']
lr_prob   = ENSEMBLE['lr']['model'].predict_proba(X_test[ENSEMBLE['lr']['cols']])[:, 1]

print('=== Baseline Comparison ===')
print(f'  Always-Positive PR-AUC : {average_precision_score(y_test, np.ones_like(y_test)):.4f}')
print(f'  LR-only PR-AUC         : {average_precision_score(y_test, lr_prob):.4f}')
print(f'  Ensemble PR-AUC        : {average_precision_score(y_test, test_prob):.4f}')

# ── Threshold sweep ───────────────────────────────────────────────────────────
thresholds = np.arange(0.05, 0.96, 0.05)
results = []
for t in thresholds:
    yp = (test_prob >= t).astype(int)
    tp = int(((yp==1)&(y_test==1)).sum()); fp = int(((yp==1)&(y_test==0)).sum())
    fn = int(((yp==0)&(y_test==1)).sum()); tn = int(((yp==0)&(y_test==0)).sum())
    prec = tp/(tp+fp+1e-9); rec = tp/(tp+fn+1e-9)
    f1   = 2*prec*rec/(prec+rec+1e-9)
    results.append(dict(threshold=round(float(t),2), precision=round(prec,4),
                        recall=round(rec,4), f1=round(f1,4),
                        accuracy=round((tp+tn)/len(y_test),4)))

df_thresh   = pd.DataFrame(results)
best_f1_row = df_thresh.loc[df_thresh['f1'].idxmax()]
print('\n=== Threshold Sweep ===')
print(f" {'Thresh':>6} {'Prec':>7} {'Recall':>7} {'F1':>7} {'Acc':>7}")
for _, row in df_thresh.iterrows():
    marker = ' ← production' if row['threshold'] == best_f1_row['threshold'] else ''
    print(f" {row['threshold']:>6.2f} {row['precision']:>7.4f} {row['recall']:>7.4f}"
          f" {row['f1']:>7.4f} {row['accuracy']:>7.4f}{marker}")

=== Baseline Comparison ===
  Always-Positive PR-AUC : 0.0886
  LR-only PR-AUC         : 0.6635
  Ensemble PR-AUC        : 0.6736

=== Threshold Sweep ===
 Thresh    Prec  Recall      F1     Acc
   0.05  0.2114  0.7536  0.3302  0.7291
   0.10  0.8065  0.7246  0.7634  0.9602
   0.15  0.8929  0.7246  0.8000  0.9679
   0.20  0.9091  0.7246  0.8065  0.9692
   0.25  0.9091  0.7246  0.8065  0.9692
   0.30  0.9259  0.7246  0.8130  0.9705
   0.35  0.9259  0.7246  0.8130  0.9705
   0.40  0.9434  0.7246  0.8197  0.9718
   0.45  0.9615  0.7246  0.8264  0.9730 ← production
   0.50  0.9615  0.7246  0.8264  0.9730
   0.55  0.9615  0.7246  0.8264  0.9730
   0.60  0.9615  0.7246  0.8264  0.9730
   0.65  0.9615  0.7246  0.8264  0.9730
   0.70  0.9592  0.6812  0.7966  0.9692
   0.75  0.9565  0.6377  0.7652  0.9653
   0.80  0.9286  0.3768  0.5361  0.9422
   0.85  0.8182  0.1304  0.2250  0.9204
   0.90  0.6667  0.0580  0.1067  0.9140
   0.95  0.0000  0.0000  0.0000  0.9089


---
## Section 9 — SHAP Explainability

SHAP (SHapley Additive exPlanations) values are computed from the XGBoost base model to provide per-feature attribution for every prediction.

Each SHAP value represents the marginal contribution of a feature to moving the prediction away from the base rate (expected value). Negative SHAP values push toward *No Stress*; positive values push toward *Stress Event*.

The top 10 features by absolute SHAP value are returned with every inference call, enabling transparent audit of any alert issued by the system.

In [16]:
_xgb_base      = ENSEMBLE['xgb']['model'].calibrated_classifiers_[0].estimator
SHAP_EXPLAINER = shap.TreeExplainer(_xgb_base)
logger.info('SHAP TreeExplainer initialised')


def explain_prediction(X_row: pd.DataFrame, top_n: int = 10) -> Dict[str, Any]:
    shap_vals = SHAP_EXPLAINER.shap_values(X_row[ENSEMBLE['xgb']['cols']])
    sv        = shap_vals[1][0] if isinstance(shap_vals, list) else shap_vals[0]
    feats     = ENSEMBLE['xgb']['cols']
    impacts   = sorted(
        [{'feature': f, 'shap_value': round(float(s), 5),
          'feature_value': round(float(X_row[f].iloc[0]), 4)}
         for f, s in zip(feats, sv)],
        key=lambda x: abs(x['shap_value']), reverse=True
    )[:top_n]
    base_val = SHAP_EXPLAINER.expected_value
    if isinstance(base_val, list): base_val = base_val[1]
    return {'feature_impacts': impacts, 'base_value': round(float(base_val), 5)}


demo_explain = explain_prediction(X_test.iloc[[0]])
print('Top SHAP features for first test sample:')
for item in demo_explain['feature_impacts'][:5]:
    print(f"  {item['feature']:<30} SHAP={item['shap_value']:+.4f}  val={item['feature_value']}")

Top SHAP features for first test sample:
  ndvi_anomaly_z                 SHAP=-1.5674  val=0.3251
  temp_dew_spread                SHAP=-0.6044  val=24.4706
  ndvi_evi_ratio                 SHAP=-0.5521  val=-0.8023
  ndvi_anomaly                   SHAP=-0.5199  val=0.0022
  temperature_2m                 SHAP=-0.5122  val=28.2603


---
## Section 10 — Uncertainty Estimation

Prediction uncertainty is quantified by measuring disagreement across the three ensemble members.

| Metric | Definition | Interpretation |
|--------|-----------|----------------|
| `mean_prob` | Weighted average of model probabilities | Primary stress score |
| `std_prob` | Standard deviation across models | Spread — higher = more disagreement |
| `confidence` | `1 − std / (mean + ε)`, clipped to [0, 1] | 1.0 = full agreement, 0.0 = maximum disagreement |
| `model_agreement` | `std < 0.10` | Boolean — used by fusion layer to route decisions |

In [17]:
def ensemble_stats(ensemble: Dict, X_row: pd.DataFrame) -> Dict[str, float]:
    """
    Compute weighted ensemble statistics for a single inference row.
    Uses ENSEMBLE_WEIGHTS to match predict_ensemble() aggregation.
    """
    individual = {
        name: float(info['model'].predict_proba(X_row[info['cols']])[:, 1][0])
        for name, info in ensemble.items()
    }
    names   = list(individual.keys())
    probs   = np.array([individual[n] for n in names])
    weights = np.array([ENSEMBLE_WEIGHTS.get(n, 1/3) for n in names])
    weights /= weights.sum()                             # normalise
    mean    = float(probs @ weights)                     # weighted mean
    std     = float(probs.std())                         # unweighted std for agreement
    conf    = float(np.clip(1 - std / (mean + 1e-9), 0, 1))
    return {
        'mean_prob'       : round(mean, 4),
        'std_prob'        : round(std, 4),
        'confidence'      : round(conf, 4),
        'model_agreement' : std < ENSEMBLE_DISAGREE,
        'individual_probs': {k: round(v, 4) for k, v in individual.items()}
    }

---
## Section 11 — Fusion Layer

The fusion layer combines the rule-based v1 engine and the ML ensemble into a single structured decision, routing each prediction through one of three branches based on ML confidence and model agreement.

### Decision Branches

| Branch | Condition | Output Source | Human Review |
|--------|-----------|--------------|-------------|
| **`v2_high_confidence`** | `mean_prob ≥ 0.55` AND models agree | ML ensemble | ❌ Not required |
| **`blended`** | `0.10 < mean_prob < 0.55` | 60% ML + 40% v1 rule score | ✅ Required |
| **`v1_fallback`** | `mean_prob ≤ 0.10` OR models disagree | v1 rule engine | ✅ Required |

### Threshold Rationale
With a 7.9% positive rate and Platt-calibrated models, calibrated probabilities for true negatives cluster in the range [0.03 – 0.17]. Setting `THRESH_LOW_CONF = 0.10` (rather than 0.35) allows the blended branch to activate for genuinely uncertain cases, rather than routing all predictions to v1 fallback.

In [18]:
def ml_to_risk_schema(prob: float, threshold: float = DECISION_THRESHOLD) -> Dict:
    if   prob >= 0.75: level = 'Critical'
    elif prob >= 0.55: level = 'High'
    elif prob >= 0.35: level = 'Moderate'
    else:              level = 'Low'
    return {'risk_level': level, 'risk_score': round(prob * 100, 1),
            'action': ACTIONS[level], 'model': 'EWS-PA-v2-ml'}


def fuse_v1_v2_protected(
    v1_result : Dict,
    v2_stats  : Dict,
    threshold : float = DECISION_THRESHOLD
) -> Dict:
    """
    3-branch fusion: high-conf ML / low-conf fallback to v1 / blend.

    v2.2 fixes:
    1. 'reasons' → 'reasoning' unified key in all output branches
    2. THRESH_LOW_CONF = 0.10 (was 0.35) to allow blend/v2 branches to activate
       given that calibrated probs on 7.9%-positive data are naturally low.
    """
    mean_prob  = v2_stats['mean_prob']
    confidence = v2_stats['confidence']
    agreement  = v2_stats['model_agreement']

    # Helper: get reasoning list regardless of key name in v1_result
    v1_reasoning = v1_result.get('reasons', v1_result.get('reasoning', []))

    # Branch 1: high confidence ML (mean_prob ≥ 0.55 AND models agree)
    if mean_prob >= THRESH_HIGH_CONF and agreement:
        schema = ml_to_risk_schema(mean_prob, threshold)
        return {**schema,
                'stress_probability': mean_prob,
                'stress_predicted'  : mean_prob >= threshold,
                'confidence'        : confidence,
                'fusion_branch'     : 'v2_high_confidence',
                'human_review'      : False,
                'v1_risk_level'     : v1_result['risk_level'],
                'reasoning'         : v1_reasoning}

    # Branch 2: low confidence → fallback to v1
    if mean_prob <= THRESH_LOW_CONF or not agreement:
        v1_positive = v1_result['risk_level'] in ['High', 'Critical']
        return {**v1_result,
                'stress_probability': round(mean_prob, 4),
                'stress_predicted'  : v1_positive,
                'confidence'        : confidence,
                'fusion_branch'     : 'v1_fallback',
                'human_review'      : True,
                'reasoning'         : v1_reasoning}

    # Branch 3: blend + human review (0.10 < mean_prob < 0.55)
    blend_prob = 0.6 * mean_prob + 0.4 * (v1_result['risk_score'] / 100)
    schema     = ml_to_risk_schema(blend_prob, threshold)
    return {**schema,
            'stress_probability': round(blend_prob, 4),
            'stress_predicted'  : blend_prob >= threshold,
            'confidence'        : confidence,
            'fusion_branch'     : 'blended',
            'human_review'      : True,
            'v1_risk_level'     : v1_result['risk_level'],
            'reasoning'         : v1_reasoning}

---
## Section 12 — End-to-End Inference Pipeline

The `predict_stress()` function is the single production entry point. It accepts a `ProtectedAreaQuery` object and returns a fully structured prediction report including risk level, fusion branch, model probabilities, rule-based reasoning, and SHAP attributions.

### Inference Steps
1. Engineer features from the data row (Section 5)
2. Run EWS v1 rule engine (Section 6)
3. Run ML ensemble + compute weighted probabilities (Section 7)
4. Estimate uncertainty across models (Section 10)
5. Route through fusion layer (Section 11)
6. Compute SHAP attributions (Section 9)
7. Return structured output dict

### Operating Modes

| Mode | Threshold | Intended Use |
|------|-----------|-------------|
| `'production'` | `DECISION_THRESHOLD` (best F2) | Automated daily monitoring |
| `'safety_audit'` | `SAFETY_THRESHOLD` (recall ≥ 0.85) | Manual expert review — high false alarm rate |

In [19]:
@dataclass
class ProtectedAreaQuery:
    """Input descriptor for a single protected area prediction."""
    query_id : str
    area_id  : int
    data_row : pd.Series


def predict_stress(
    query     : ProtectedAreaQuery,
    ensemble  : Optional[Dict]  = None,
    threshold : Optional[float] = None,
    mode      : str             = 'production'  # 'production' | 'safety_audit'
) -> Dict[str, Any]:
    """
    Full inference pipeline for a single protected area query.
    mode='production'    → DECISION_THRESHOLD (best-F2)
    mode='safety_audit'  → SAFETY_THRESHOLD (recall≥0.85) — high false alarm rate
    """
    if ensemble  is None: ensemble  = ENSEMBLE
    if threshold is None:
        threshold = SAFETY_THRESHOLD if mode == 'safety_audit' else DECISION_THRESHOLD
    if mode == 'safety_audit':
        logger.warning('SAFETY AUDIT MODE — high false alarm rate (Precision~0.09)')

    t_start = dt.datetime.now()

    feats   = engineer_features_protected(query.data_row)
    X_infer = pd.DataFrame([feats])[FEATURE_NAMES].fillna(0.0)
    for col in X_train.columns:
        if col not in X_infer.columns:
            X_infer[col] = 0.0
    X_infer = X_infer[X_train.columns]

    v1_result = predict_risk_v1_protected(feats)
    stats     = ensemble_stats(ensemble, X_infer)
    final     = fuse_v1_v2_protected(v1_result, stats, threshold)

    try:
        explanation = explain_prediction(X_infer)
    except Exception as exc:
        logger.warning('SHAP failed: %s', exc)
        explanation = {'feature_impacts': [], 'base_value': 0.0}

    elapsed_ms = (dt.datetime.now() - t_start).total_seconds() * 1000

    return {
        'query_id'           : query.query_id,
        'timestamp'          : dt.datetime.utcnow().isoformat() + 'Z',
        'area_id'            : query.area_id,
        'area_name'          : PROTECTED_AREAS.get(query.area_id, '?'),
        'mode'               : mode,
        'stress_predicted'   : final['stress_predicted'],
        'stress_probability' : final['stress_probability'],
        'risk_level'         : final['risk_level'],
        'risk_score'         : final['risk_score'],
        'action'             : final['action'],
        'confidence'         : final['confidence'],
        'human_review'       : final['human_review'],
        'fusion_branch'      : final['fusion_branch'],
        'individual_probs'   : stats['individual_probs'],
        'reasoning'          : final.get('reasoning', []),
        'shap_explanation'   : explanation,
        'inference_ms'       : round(elapsed_ms, 1),
        'model_version'      : 'EWS-PA-v2.3'
    }

---
## Section 13 — Artifact Persistence

Trained models and configuration are serialised to disk after each training run. This allows inference to be performed without re-training.

### Saved Artifacts

| File | Content |
|------|---------|
| `xgb_model.joblib` | Calibrated XGBoost classifier |
| `rf_model.joblib` | Calibrated Random Forest classifier |
| `lr_model.joblib` | Calibrated Logistic Regression pipeline |
| `{name}_cols.json` | Feature column list for each model |
| `config.json` | Thresholds, feature names, model version, timestamp |

In [20]:
def save_artifacts(ensemble, threshold, safety_threshold,
                   feature_names, lr_safe_features, out_dir=ARTIFACTS_DIR):
    out_dir.mkdir(exist_ok=True)
    for name, info in ensemble.items():
        joblib.dump(info['model'], out_dir / f'{name}_model.joblib')
        with open(out_dir / f'{name}_cols.json', 'w') as fh:
            json.dump(info['cols'], fh)
    with open(out_dir / 'config.json', 'w', encoding='utf-8') as fh:
        json.dump({
            'decision_threshold': threshold,
            'safety_threshold'  : safety_threshold,
            'feature_names'     : feature_names,
            'lr_safe_features'  : lr_safe_features,
            'model_version'     : 'EWS-PA-v2.3',
            'saved_at'          : dt.datetime.utcnow().isoformat() + 'Z'
        }, fh, indent=2, ensure_ascii=False)
    logger.info('Artifacts saved to %s', out_dir.resolve())


def load_artifacts(out_dir=ARTIFACTS_DIR):
    with open(out_dir / 'config.json', encoding='utf-8') as fh:
        cfg = json.load(fh)
    ens = {}
    for name in ['xgb', 'rf', 'lr']:
        mp = out_dir / f'{name}_model.joblib'
        cp = out_dir / f'{name}_cols.json'
        if mp.exists():
            with open(cp) as fh: cols = json.load(fh)
            ens[name] = {'model': joblib.load(mp), 'cols': cols}
    return ens, cfg['decision_threshold'], cfg['feature_names']


save_artifacts(ENSEMBLE, DECISION_THRESHOLD, SAFETY_THRESHOLD,
               FEATURE_NAMES, LR_SAFE_FEATURES)
print('Artifacts saved to:', ARTIFACTS_DIR.resolve())

Artifacts saved to: /content/drive/MyDrive/data_EWS/artifacts_protected


In [21]:
# ── تحميل الموديلات المحفوظة سابقًا بدلاً من إعادة التدريب ──────────────────
# شغّل الخلية دي بدل خلايا section 7 (train_ensemble) لو عندك تدريب سابق محفوظ.

ENSEMBLE, DECISION_THRESHOLD, FEATURE_NAMES = load_artifacts(ARTIFACTS_DIR)

if ENSEMBLE:
    print(f'تم تحميل {len(ENSEMBLE)} موديل(ات) من: {ARTIFACTS_DIR.resolve()}')
    print('الموديلات المحمّلة:', list(ENSEMBLE.keys()))
    print('عتبة القرار (decision_threshold):', DECISION_THRESHOLD)
else:
    print('⚠ لا توجد artifacts محفوظة في هذا المسار بعد — شغّل أقسام التدريب (3 → 7) أولًا، '
          'ثم نفّذ خلية save_artifacts في Section 13.')

تم تحميل 3 موديل(ات) من: /content/drive/MyDrive/data_EWS/artifacts_protected
الموديلات المحمّلة: ['xgb', 'rf', 'lr']
عتبة القرار (decision_threshold): 0.69138677431065


---
## Section 14 — Demo: Single & Batch Predictions

Demonstrates the full inference pipeline on real records from the dataset.

**Single prediction:** Wadi El-Rayan protected area (area_id = 27) — production mode.

**Batch prediction:** Five geographically diverse protected areas covering desert, coastal, and mountainous ecosystems.

**Branch distribution check:** Samples 100 records from the test set to verify that all three fusion branches are active — confirming the weighted ensemble and threshold calibration are working correctly.

In [22]:
# ── Single prediction demo — وادي الريان ────────────────────────────────────
demo_row   = LABELLED_DF[LABELLED_DF['area_id'] == 27].iloc[100]
demo_query = ProtectedAreaQuery(query_id='DEMO-001', area_id=27, data_row=demo_row)
result     = predict_stress(demo_query)   # production mode

print('\n' + '='*62)
print(' EWS Protected Areas — Vegetation Stress Report')
print('='*62)
print(f"  Query ID             : {result['query_id']}")
print(f"  Area                 : {result['area_name']} (ID={result['area_id']})")
print(f"  Mode                 : {result['mode']}")
print(f"  Timestamp            : {result['timestamp']}")
print()
print(f"  ⚠  Stress Predicted  : {result['stress_predicted']}")
print(f"  📊 Probability        : {result['stress_probability']:.4f}")
print(f"  🔴 Risk Level         : {result['risk_level']}")
print(f"  📈 Risk Score         : {result['risk_score']}")
print(f"  🎯 Confidence         : {result['confidence']:.4f}")
print(f"  👤 Human Review       : {result['human_review']}")
print(f"  🔀 Fusion Branch      : {result['fusion_branch']}")
print(f"  ⏱  Inference time     : {result['inference_ms']} ms")
print()
print(f"  📋 Action: {result['action']}")
print()
print('  Model probabilities:')
for k, v in result['individual_probs'].items():
    print(f'    {k.upper():<6} : {v:.4f}')
print()
reasoning = result.get('reasoning', [])
if reasoning:
    print('  Rule-based reasoning:')
    for r in reasoning:
        print(f'    • {r}')
else:
    print('  Rule-based reasoning: (none triggered — low stress conditions)')
print()
print('  Top SHAP features:')
for item in result['shap_explanation']['feature_impacts'][:5]:
    direction = '▲' if item['shap_value'] > 0 else '▼'
    print(f"    {direction} {item['feature']:<30} SHAP={item['shap_value']:+.4f}")
print('='*62)


 EWS Protected Areas — Vegetation Stress Report
  Query ID             : DEMO-001
  Area                 : محمية وادي الريان (ID=27)
  Mode                 : production
  Timestamp            : 2026-06-20T19:52:10.892892Z

  ⚠  Stress Predicted  : False
  📊 Probability        : 0.0546
  🔴 Risk Level         : Moderate
  📈 Risk Score         : 40.0
  🎯 Confidence         : 0.0000
  👤 Human Review       : True
  🔀 Fusion Branch      : v1_fallback
  ⏱  Inference time     : 153.4 ms

  📋 Action: Increase monitoring frequency. Investigate potential stressors.

  Model probabilities:
    XGB    : 0.0315
    RF     : 0.0251
    LR     : 0.1564

  Rule-based reasoning:
    • NDVI 0.038 below stress threshold (0.15) [+20]
    • Soil moisture 0.005 m³/m³ below area baseline (P20=0.009) [+10]
    • Multi-stress co-occurrence (composite=0.60) [+10]

  Top SHAP features:
    ▼ ndvi_anomaly_z                 SHAP=-1.8371
    ▼ precip_zscore                  SHAP=-0.7490
    ▼ temperature_2m        

In [23]:
# ── Batch demo: 5 areas + fusion branch distribution ─────────────────────────
DEMO_AREAS = [3, 6, 12, 14, 27]  # الجلف الكبير، الصحراء البيضاء، رأس محمد، سانت كاثرين، وادي الريان

print(f"{'Query':<12} {'Area':<35} {'Prob':>7} {'Level':<10} {'Branch'}")
print('-' * 82)
for i, area_id in enumerate(DEMO_AREAS):
    area_df = LABELLED_DF[LABELLED_DF['area_id'] == area_id]
    if area_df.empty: continue
    row = area_df.iloc[len(area_df)//2]
    q   = ProtectedAreaQuery(f'BATCH-{i:03d}', area_id, row)
    r   = predict_stress(q)
    print(f"{r['query_id']:<12} {r['area_name']:<35} "
          f"{r['stress_probability']:>7.4f} {r['risk_level']:<10} {r['fusion_branch']}")

# ── Branch distribution across full test set ──────────────────────────────────
print('\n=== Fusion Branch Distribution (test set sample, n=100) ===')
test_df   = LABELLED_DF.iloc[sort_idx[-779:]].reset_index(drop=True)
sample_n  = min(100, len(test_df))
branches  = []
for _, row in test_df.sample(sample_n, random_state=42).iterrows():
    q = ProtectedAreaQuery('tmp', int(row['area_id']), row)
    r = predict_stress(q)
    branches.append(r['fusion_branch'])
print(pd.Series(branches).value_counts().to_string())
print('\n✅ If blended/v2_high_confidence appear → fusion thresholds fix worked')

Query        Area                                   Prob Level      Branch
----------------------------------------------------------------------------------
BATCH-000    محمية الجلف الكبير                   0.0441 Low        v1_fallback
BATCH-001    محمية الصحراء البيضاء                0.7633 Moderate   v1_fallback
BATCH-002    محمية رأس محمد                       0.0305 Low        v1_fallback
BATCH-003    محمية سانت كاثرين                    0.0552 Low        v1_fallback
BATCH-004    محمية وادي الريان                    0.7237 Moderate   v1_fallback

=== Fusion Branch Distribution (test set sample, n=100) ===
v1_fallback           97
blended                2
v2_high_confidence     1

✅ If blended/v2_high_confidence appear → fusion thresholds fix worked


---
## Section 15 — Production Safeguards: Input Validation & Prediction Logging

Two additions required before any production deployment:

### 15a — Input Validation (`validate_query_input`)
Catches malformed inputs **before** they reach the model, preventing silent errors or garbage predictions.

| Check | Rule | Action on Failure |
|-------|------|-------------------|
| `area_id` range | Must be 0–30 | Raise `ValueError` |
| Required columns | All 13 raw columns must be present | Raise `ValueError` with missing list |
| Temperature sanity | `temperature_2m` must be in [250, 330] K | Log warning + flag |
| NDVI sanity | `ndvi_mean` must be in [−0.5, 1.0] | Log warning + flag |
| Null values | No NaN in required columns | Raise `ValueError` with null list |

### 15b — Prediction Logging (`log_prediction`)
Writes every prediction to a JSONL file on Google Drive. Each line is one prediction — machine-readable and appendable.

**Why JSONL?** Each prediction is a self-contained JSON object on one line. Easy to `grep`, easy to load with `pd.read_json(..., lines=True)`, and safe to append without rewriting the file.

**Logged fields:** `query_id`, `timestamp`, `area_id`, `area_name`, `stress_predicted`, `stress_probability`, `risk_level`, `fusion_branch`, `mode`, `inference_ms`, `model_version`, plus an optional `ground_truth` field for expert feedback.

### 15c — Safe Predict Wrapper (`predict_stress_safe`)
Wraps `predict_stress()` with validation + logging + full error handling. This is the function to call in production — not `predict_stress()` directly.


In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 15 — Production Safeguards: Input Validation & Prediction Logging
# EWS Protected Areas v2.4
# ══════════════════════════════════════════════════════════════════════════════

import traceback

# ── Log file path (Google Drive) ──────────────────────────────────────────────
LOG_DIR  = Path('/content/drive/MyDrive/data_EWS/logs')
LOG_FILE = LOG_DIR / 'predictions.jsonl'

# Required raw columns that must exist in data_row before feature engineering
REQUIRED_COLUMNS = [
    'ndvi_mean', 'savi', 'evi', 'vegetation_percent',
    'temperature_2m', 'total_precipitation_sum',
    'volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_2',
    'volumetric_soil_water_layer_3', 'volumetric_soil_water_layer_4',
    'surface_solar_radiation_downwards_sum',
    'u_component_of_wind_10m', 'v_component_of_wind_10m',
]

# ── 15a: Input Validation ─────────────────────────────────────────────────────
def validate_query_input(query: 'ProtectedAreaQuery') -> Dict[str, Any]:
    """
    Validate a ProtectedAreaQuery before it reaches the model.
    Returns a validation report dict.
    Raises ValueError for hard failures (bad area_id, missing columns, nulls).
    Logs warnings for soft failures (out-of-range values) and adds flags.
    """
    report = {'valid': True, 'warnings': [], 'flags': []}
    row    = query.data_row

    # ── Hard check 1: area_id must be a known protected area ──────────────────
    if query.area_id not in PROTECTED_AREAS:
        raise ValueError(
            f'area_id={query.area_id} is not valid. '
            f'Must be one of: {sorted(PROTECTED_AREAS.keys())}'
        )

    # ── Hard check 2: required columns must all be present ────────────────────
    if isinstance(row, pd.Series):
        available = set(row.index)
    else:
        available = set(row.keys())

    missing = [c for c in REQUIRED_COLUMNS if c not in available]
    if missing:
        raise ValueError(
            f'Missing required columns for query {query.query_id}: {missing}'
        )

    # ── Hard check 3: no NaN in required columns ──────────────────────────────
    nulls = [c for c in REQUIRED_COLUMNS if pd.isna(row.get(c, None))]
    if nulls:
        raise ValueError(
            f'NaN values in required columns for query {query.query_id}: {nulls}'
        )

    # ── Soft check 1: temperature sanity (ERA5 in Kelvin → expect 250–330 K) ──
    temp_k = float(row.get('temperature_2m', 295.0))
    if not (250.0 <= temp_k <= 330.0):
        msg = (
            f'[{query.query_id}] temperature_2m={temp_k:.1f} K is outside '
            f'expected range [250, 330] K — possible unit error or sensor fault'
        )
        logger.warning(msg)
        report['warnings'].append(msg)
        report['flags'].append('temperature_out_of_range')

    # ── Soft check 2: NDVI sanity (valid range −0.5 to 1.0) ──────────────────
    ndvi = float(row.get('ndvi_mean', 0.0))
    if not (-0.5 <= ndvi <= 1.0):
        msg = (
            f'[{query.query_id}] ndvi_mean={ndvi:.4f} is outside '
            f'valid range [−0.5, 1.0] — possible satellite artifact'
        )
        logger.warning(msg)
        report['warnings'].append(msg)
        report['flags'].append('ndvi_out_of_range')

    # ── Soft check 3: soil moisture sanity (0 to 1 m³/m³) ───────────────────
    sm1 = float(row.get('volumetric_soil_water_layer_1', 0.0))
    if not (-0.01 <= sm1 <= 1.0):
        msg = (
            f'[{query.query_id}] soil_moisture_l1={sm1:.4f} is outside '
            f'valid range [0, 1] m³/m³'
        )
        logger.warning(msg)
        report['warnings'].append(msg)
        report['flags'].append('soil_moisture_out_of_range')

    report['valid'] = True  # passed all hard checks
    return report


# ── 15b: Prediction Logging ───────────────────────────────────────────────────
def log_prediction(
    result       : Dict[str, Any],
    validation   : Dict[str, Any],
    ground_truth : Optional[int] = None,   # 1=stress confirmed, 0=no stress, None=unknown
    log_file     : Path = LOG_FILE
) -> None:
    """
    Append one prediction to the JSONL log file on Google Drive.

    File format: one JSON object per line (JSONL).
    Load later with: pd.read_json('predictions.jsonl', lines=True)

    ground_truth: fill in after field verification.
        1  = stress confirmed by expert
        0  = no stress (false alarm)
        None = not yet verified
    """
    try:
        log_file.parent.mkdir(parents=True, exist_ok=True)

        record = {
            # ── Identity ────────────────────────────────────────────────────
            'query_id'           : result.get('query_id'),
            'timestamp'          : result.get('timestamp'),
            'model_version'      : result.get('model_version'),
            # ── Location ────────────────────────────────────────────────────
            'area_id'            : result.get('area_id'),
            'area_name'          : result.get('area_name'),
            # ── Prediction ──────────────────────────────────────────────────
            'mode'               : result.get('mode'),
            'stress_predicted'   : result.get('stress_predicted'),
            'stress_probability' : result.get('stress_probability'),
            'risk_level'         : result.get('risk_level'),
            'risk_score'         : result.get('risk_score'),
            'fusion_branch'      : result.get('fusion_branch'),
            'human_review'       : result.get('human_review'),
            'confidence'         : result.get('confidence'),
            # ── Model internals ─────────────────────────────────────────────
            'prob_xgb'           : result.get('individual_probs', {}).get('xgb'),
            'prob_rf'            : result.get('individual_probs', {}).get('rf'),
            'prob_lr'            : result.get('individual_probs', {}).get('lr'),
            'inference_ms'       : result.get('inference_ms'),
            # ── Validation flags ────────────────────────────────────────────
            'validation_warnings': validation.get('warnings', []),
            'validation_flags'   : validation.get('flags', []),
            # ── Ground truth (filled by expert after field visit) ────────────
            'ground_truth'       : ground_truth,
        }

        with open(log_file, 'a', encoding='utf-8') as fh:
            fh.write(json.dumps(record, ensure_ascii=False) + '\n')

        logger.info(
            'Logged prediction %s | area=%s | stress=%s | prob=%.3f | branch=%s',
            record['query_id'], record['area_name'],
            record['stress_predicted'], record['stress_probability'],
            record['fusion_branch']
        )

    except Exception as exc:
        # Logging failure must NEVER crash the inference pipeline
        logger.error('Failed to log prediction %s: %s', result.get('query_id'), exc)


# ── 15c: Safe Predict Wrapper ─────────────────────────────────────────────────
def predict_stress_safe(
    query        : 'ProtectedAreaQuery',
    ensemble     : Optional[Dict]  = None,
    threshold    : Optional[float] = None,
    mode         : str             = 'production',
    ground_truth : Optional[int]   = None,
    do_log       : bool            = True,
) -> Dict[str, Any]:
    """
    Production entry point — wraps predict_stress() with:
      1. Input validation  (raises on hard errors, warns on soft)
      2. Full inference
      3. Prediction logging to Google Drive JSONL
      4. Structured error handling (returns error dict instead of crashing)

    Use this function in production. Do NOT call predict_stress() directly.

    Parameters
    ----------
    query        : ProtectedAreaQuery with area_id and data_row
    ensemble     : trained ensemble dict (defaults to global ENSEMBLE)
    threshold    : decision threshold (defaults to DECISION_THRESHOLD)
    mode         : 'production' | 'safety_audit'
    ground_truth : 1/0/None — fill after field verification
    do_log       : set False to skip logging (e.g. in unit tests)
    """
    # ── Step 1: Validate ──────────────────────────────────────────────────────
    try:
        validation = validate_query_input(query)
    except ValueError as ve:
        logger.error('Input validation FAILED for %s: %s', query.query_id, ve)
        return {
            'query_id'   : query.query_id,
            'area_id'    : query.area_id,
            'timestamp'  : dt.datetime.utcnow().isoformat() + 'Z',
            'status'     : 'validation_error',
            'error'      : str(ve),
            'stress_predicted' : None,
        }

    # ── Step 2: Inference ─────────────────────────────────────────────────────
    try:
        result = predict_stress(
            query     = query,
            ensemble  = ensemble,
            threshold = threshold,
            mode      = mode,
        )
        result['status'] = 'ok'

    except Exception as exc:
        logger.error(
            'Inference FAILED for %s: %s\n%s',
            query.query_id, exc, traceback.format_exc()
        )
        return {
            'query_id'   : query.query_id,
            'area_id'    : query.area_id,
            'timestamp'  : dt.datetime.utcnow().isoformat() + 'Z',
            'status'     : 'inference_error',
            'error'      : str(exc),
            'stress_predicted' : None,
        }

    # ── Step 3: Log ───────────────────────────────────────────────────────────
    if do_log:
        log_prediction(result, validation, ground_truth)

    return result


# ── Utility: load prediction log ──────────────────────────────────────────────
def load_prediction_log(log_file: Path = LOG_FILE) -> pd.DataFrame:
    """
    Load the full prediction log as a DataFrame.
    Use this to analyse model performance over time or fill in ground_truth.

    Example:
        df = load_prediction_log()
        # Fill ground truth for a confirmed stress event:
        # Edit the JSONL file directly, or re-log with ground_truth=1
        print(df[['query_id','area_name','stress_predicted','ground_truth']].tail(10))
    """
    if not log_file.exists():
        logger.warning('Log file not found: %s', log_file)
        return pd.DataFrame()
    return pd.read_json(log_file, lines=True)


logger.info('Section 15 loaded — input validation + prediction logging ready')
print('Section 15 loaded successfully.')
print(f'  Log file path : {LOG_FILE}')
print(f'  Required cols : {len(REQUIRED_COLUMNS)} columns checked')
print()
print('Production entry point: predict_stress_safe(query, mode="production")')
print('Direct use:             predict_stress() — no validation, no logging')


Section 15 loaded successfully.
  Log file path : /content/drive/MyDrive/data_EWS/logs/predictions.jsonl
  Required cols : 13 columns checked

Production entry point: predict_stress_safe(query, mode="production")
Direct use:             predict_stress() — no validation, no logging


In [25]:
# ── Section 15 Demo: validate + safe predict ─────────────────────────────────
print('=== Section 15 Demo ===')
print()

# ── Demo 1: Valid query ───────────────────────────────────────────────────────
print('--- Demo 1: Valid query (محمية وادي الريان) ---')
demo_row   = LABELLED_DF[LABELLED_DF['area_id'] == 27].iloc[100]
demo_query = ProtectedAreaQuery(query_id='SAFE-001', area_id=27, data_row=demo_row)

# Validate only
val_report = validate_query_input(demo_query)
print(f'  Validation passed: {val_report["valid"]}')
print(f'  Warnings         : {val_report["warnings"] or "none"}')
print(f'  Flags            : {val_report["flags"] or "none"}')
print()

# Safe predict (do_log=False to avoid writing to Drive in demo)
safe_result = predict_stress_safe(demo_query, do_log=False)
print(f'  Status           : {safe_result["status"]}')
print(f'  Stress predicted : {safe_result["stress_predicted"]}')
print(f'  Probability      : {safe_result["stress_probability"]:.4f}')
print(f'  Risk level       : {safe_result["risk_level"]}')
print(f'  Fusion branch    : {safe_result["fusion_branch"]}')
print()

# ── Demo 2: Invalid area_id ───────────────────────────────────────────────────
print('--- Demo 2: Invalid area_id (99) ---')
bad_query = ProtectedAreaQuery(query_id='SAFE-002', area_id=99, data_row=demo_row)
bad_result = predict_stress_safe(bad_query, do_log=False)
print(f'  Status : {bad_result["status"]}')
print(f'  Error  : {bad_result["error"]}')
print()

# ── Demo 3: Out-of-range temperature (soft warning) ──────────────────────────
print('--- Demo 3: Suspicious temperature (999 K) — soft warning ---')
bad_temp_row          = demo_row.copy()
bad_temp_row['temperature_2m'] = 999.0   # obviously wrong
warn_query = ProtectedAreaQuery(query_id='SAFE-003', area_id=27, data_row=bad_temp_row)
warn_result = predict_stress_safe(warn_query, do_log=False)
print(f'  Status           : {warn_result["status"]}')
print(f'  Stress predicted : {warn_result["stress_predicted"]}')
print(f'  (prediction still runs — temperature warning is soft, not a hard stop)')
print()

# ── Demo 4: Log format preview ────────────────────────────────────────────────
print('--- Demo 4: Log record preview (not written to disk) ---')
import io
safe_result2 = predict_stress_safe(
    ProtectedAreaQuery('SAFE-004', 27, demo_row),
    do_log=False
)
val2 = validate_query_input(ProtectedAreaQuery('SAFE-004', 27, demo_row))
log_record = {
    'query_id'          : safe_result2.get('query_id'),
    'timestamp'         : safe_result2.get('timestamp'),
    'area_name'         : safe_result2.get('area_name'),
    'stress_predicted'  : safe_result2.get('stress_predicted'),
    'stress_probability': safe_result2.get('stress_probability'),
    'risk_level'        : safe_result2.get('risk_level'),
    'fusion_branch'     : safe_result2.get('fusion_branch'),
    'ground_truth'      : None,
    'validation_flags'  : val2.get('flags', []),
}
print('  JSONL record (one line per prediction):')
print(' ', json.dumps(log_record, ensure_ascii=False))
print()
print('=== Section 15 Demo Complete ===')
print()
print('In production, call:')
print('  result = predict_stress_safe(query, mode="production", do_log=True)')
print('  # ground_truth=1 after field expert confirms stress event')
print('  result = predict_stress_safe(query, ground_truth=1, do_log=True)')


ERROR:ews_protected_areas:Input validation FAILED for SAFE-002: area_id=99 is not valid. Must be one of: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]


=== Section 15 Demo ===

--- Demo 1: Valid query (محمية وادي الريان) ---
  Validation passed: True
  Warnings         : none
  Flags            : none

  Status           : ok
  Stress predicted : False
  Probability      : 0.0546
  Risk level       : Moderate
  Fusion branch    : v1_fallback

--- Demo 2: Invalid area_id (99) ---
  Status : validation_error
  Error  : area_id=99 is not valid. Must be one of: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]

--- Demo 3: Suspicious temperature (999 K) — soft warning ---
  Status           : ok
  Stress predicted : True
  (prediction still runs — temperature warning is soft, not a hard stop)

--- Demo 4: Log record preview (not written to disk) ---
  JSONL record (one line per prediction):
  {"query_id": "SAFE-004", "timestamp": "2026-06-20T19:52:22.113163Z", "area_name": "محمية وادي الريان", "stress_predicted": false, "stress_probability": 0.0546, "risk_level": "Moderate",

---
## Notebook Summary

| Section | Function / Output | Key Design Decision |
|---------|-------------------|---------------------|
| 1. Setup | Imports, logging, Google Drive mount | `statsmodels` included for VIF |
| 2. Config | All constants, thresholds, model params, `ENSEMBLE_WEIGHTS` | Weighted 0.5 / 0.3 / 0.2 for XGB / RF / LR |
| 3. Data Loading | `load_protected_areas_data()` → 31 CSVs | Filename-based area ID extraction |
| 3b. Baselines | `AREA_PRECIP_BASELINES`, `AREA_PRECIP_STDS`, `AREA_SOILMOIST_P20` | Per-area statistics from raw data |
| 4. Label Generation | `generate_ndvi_stress_labels()` | Observed NDVI anomaly — not rule-generated |
| 5. Feature Engineering | `engineer_features_protected()` — 55 features | Kelvin→°C · anomaly-based drought & water flags |
| 5b. VIF Check | `compute_vif()` → `LR_SAFE_FEATURES` | Removes features with VIF > 10 from LR only |
| 6. EWS v1 Rules | `predict_risk_v1_protected()` | Anomaly-based flags · Critical ≥ 80 pts |
| 7. ML Ensemble | `train_ensemble()` · `predict_ensemble()` | Weighted mean · `scale_pos_weight` computed locally |
| 8. Evaluation | `eval_ensemble()` · F2-optimised threshold | Dual-threshold: production vs safety audit |
| 9. SHAP | `explain_prediction()` | XGBoost TreeExplainer · top-10 feature attributions |
| 10. Uncertainty | `ensemble_stats()` | Weighted mean · confidence from model spread |
| 11. Fusion | `fuse_v1_v2_protected()` — 3-branch logic | `reasoning` key unified · thresholds tuned for imbalance |
| 12. Pipeline | `predict_stress(query, mode)` | Single entry point for production & audit modes |
| 13. Persistence | `save_artifacts()` / `load_artifacts()` | Per-model feature column lists saved |
| 14. Demo | Single + batch + branch distribution | Verifies all three fusion branches are active |

---

## System Design Decisions

### Label Methodology
Stress labels are derived from an observed Landsat NDVI signal using anomaly detection — not from hand-crafted weather rules. This methodology is grounded in peer-reviewed literature:

- Pettorelli et al. (2005). *The Normalized Difference Vegetation Index.* Trends in Ecology & Evolution.
- Tucker et al. (2005). An extended AVHRR 8-km NDVI dataset compatible with MODIS and SPOT vegetation NDVI data. *International Journal of Remote Sensing.*
- UNEP (2006). *Global Deserts Outlook.*

### Handling Class Imbalance
The dataset has a 7.9% stress event rate. Three complementary strategies address this:
1. `scale_pos_weight` in XGBoost and `class_weight='balanced'` in RF and LR
2. F2-score threshold optimisation (recall-weighted)
3. Weighted ensemble aggregation to suppress LR probability inflation

### Desert-Aware Feature Engineering
Absolute stress thresholds designed for croplands are inappropriate for arid protected areas where low precipitation and low soil moisture are chronic baseline conditions, not anomalies. All stress flags are computed relative to per-area historical baselines derived from 40+ years of ERA5 data.

### Fusion Architecture
The three-branch fusion layer ensures that ML uncertainty does not silently suppress alerts. When the ML ensemble is uncertain (`mean_prob ≤ 0.10`), the system falls back to the interpretable rule engine rather than returning a low-confidence automated decision.